# CAMARA Fall25 RAG — Quickstart Notebook

**Open Telco RAG**

This notebook walks through the Open Telco RAG pipeline for the
**CAMARA Fall25** corpus — 75 API repositories from the
[camaraproject](https://github.com/camaraproject) GitHub organisation.

By the end of this notebook you will have:
- Ingested 75 CAMARA API repos (YAML + Markdown) into a local ChromaDB
- Run semantic search across chunks
- Discovered 3GPP cross-references from CAMARA answers
- Evaluated retrieval quality with the evaluation metrics

---

### Prerequisites

```bash
git clone https://github.com/Znbne/telcolens
cd open-telco-rag
pip install -e .
```

Clone CAMARA repositories:
```bash
python scripts/clone_camara.py --output data/camara
```

Set your API keys in a `.env` file:
```
# For answer generation (choose one or both)
ANTHROPIC_API_KEY=sk-ant-...        # Claude models direct
OPENROUTER_API_KEY=sk-or-...        # Any model via OpenRouter
```

## 1. Setup

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from collections import Counter
import statistics

load_dotenv()

DATA_DIR   = Path("data/camara")
DB_PATH    = "chroma_db"
MODEL_KEY  = "minilm"
COLLECTION = f"camara_{MODEL_KEY}"

print(f"Data directory : {DATA_DIR}")
print(f"ChromaDB path  : {DB_PATH}")
print(f"Embedding model: {MODEL_KEY}")
print(f"Collection     : {COLLECTION}")

## 2. Check available repositories

CAMARA distributes each API as an independent GitHub repository.
The corpus includes 75 API repos after excluding 8 non-specification repos
(.github, camaraproject.github.io, EasyCLA, WorkingGroups, test repos,
Template_PI_Repository).

In [ ]:
if DATA_DIR.exists():
    repos = [d for d in DATA_DIR.iterdir() if d.is_dir()]
    yaml_files = list(DATA_DIR.rglob("*.yaml")) + list(DATA_DIR.rglob("*.yml"))
    md_files   = list(DATA_DIR.rglob("*.md"))
    print(f"Repositories cloned : {len(repos)}")
    print(f"YAML/OpenAPI files  : {len(yaml_files)}")
    print(f"Markdown files      : {len(md_files)}")
    print("\nSample repositories:")
    for r in sorted(repos)[:8]:
        print(f"  {r.name}")
    print("  ...")
else:
    print(f"[INFO] {DATA_DIR} not found.")
    print("Clone CAMARA repos: python scripts/clone_camara.py --output data/camara")

## 3. Ingest into ChromaDB

The CAMARA ingest pipeline:
1. Loads YAML specifications and **converts to natural language prose**
   (endpoint summaries, parameter descriptions, error codes, schemas)
2. Loads Markdown documentation (README, user stories, meeting minutes)
3. Excludes non-technical files (CHANGELOG, LICENSE, CODEOWNERS, etc.)
4. Applies heading-aware chunking for Markdown files
5. Embeds and stores with API repository name in metadata

**Key**: Raw YAML syntax is never indexed — only the natural language
content embedded within the OpenAPI specification is ingested.

In [ ]:
from open_telco_rag.sources.camara import CAMARASource
from open_telco_rag.embedders.sentense_transformers import SentenceTransformerEmbedder
from open_telco_rag.stores.chroma import ChromaStore
from open_telco_rag.core.pipeline import ingest

source   = CAMARASource()
embedder = SentenceTransformerEmbedder(model_key=MODEL_KEY)
store    = ChromaStore(path=DB_PATH, collection_name=COLLECTION)

print(f"Starting ingest — {store.count()} chunks already in collection")

if DATA_DIR.exists():
    stats = ingest(
        source     = source,
        store      = store,
        embedder   = embedder,
        data_dir   = str(DATA_DIR),
        chunk_size = 1024,
        overlap    = 200,
        resume     = True,
    )
    print(f"\nIngest complete")
    print(f"  Files processed : {stats.files_processed}")
    print(f"  Chunks stored   : {stats.chunks_stored}")
    print(f"  Skipped         : {stats.skipped_files}")
    print(f"  Errors          : {stats.error_files}")
print(f"  Total in DB     : {store.count():,}")

## 4. Explore the Collection

CAMARA metadata includes the repository (API) name and file type.
Each repository corresponds to one API family.

In [ ]:
import chromadb

client     = chromadb.PersistentClient(path=DB_PATH)
collection = client.get_collection(COLLECTION)
sample     = collection.get(limit=2000, include=["documents", "metadatas"])

repo_counts = Counter(
    m.get("source", "?").split("/")[0] for m in sample["metadatas"]
)
type_counts = Counter(m.get("file_type", "?") for m in sample["metadatas"])
lengths     = [len(d) for d in sample["documents"]]

print(f"Collection: {COLLECTION}")
print(f"Total chunks: {collection.count():,}\n")
print(f"Top API repos (sample of 2,000):")
print(f"{'API Repository':<40} {'Chunks':>8}")
print("-" * 50)
for repo, count in repo_counts.most_common(12):
    print(f"{repo:<40} {count:>8}")

print(f"\nFile types:")
for ft, count in type_counts.most_common():
    print(f"  {ft:<20} {count}")

print(f"\nChunk length statistics (chars):")
print(f"  Mean   : {statistics.mean(lengths):.0f}")
print(f"  Median : {statistics.median(lengths):.0f}")

## 5. Semantic Search

CAMARA chunks contain machine-precise technical content — error codes,
endpoint names, parameter constraints — extracted from OpenAPI YAML.

In [ ]:
from open_telco_rag.core.pipeline import search

QUERY = "What is the Quality on Demand API and what QoS profiles does it support?"
TOP_K = 5

results = search(
    QUERY,
    store    = store,
    embedder = embedder,
    top_k    = TOP_K,
)

print(f"Query: '{QUERY}'")
print(f"Top-{TOP_K} results:\n")
print(f"{'Rank':<5} {'Similarity':>10} {'Source (truncated)'}")
print("-" * 70)

for i, r in enumerate(results, 1):
    sim = round(1 - r.distance, 3)
    src = r.metadata.get("source", "")[:55]
    print(f"{i:<5} {sim:>10.3f} {src}")

print(f"\nTop result preview:")
print(results[0].text[:400])

### 5.1 Cross-standard reference discovery

CAMARA specs explicitly reference the 3GPP standards they build upon.
Answers to CAMARA questions automatically surface which 3GPP series to
query for deeper technical detail — enabling multi-hop retrieval.

In [ ]:
from open_telco_rag.generators.openrouter import OpenRouterGenerator
generator = OpenRouterGenerator(model="meta-llama/llama-3.3-70b-instruct")

import re

QUERY2 = "How does CAMARA handle network slice booking for QoS?"

results2  = search(QUERY2, store=store, embedder=embedder, top_k=8)

print(f"Query: '{QUERY2}'")
print(f"\nSources retrieved ({len(results2)} chunks from {len({r.metadata.get('source','') for r in results2})} documents):")
for src in sorted({r.metadata.get('source', '') for r in results2}):
    print(f"  {src}")

from open_telco_rag.core.pipeline import generate
response2 = generate(QUERY2, results2, source_type="camara", generator=generator)
print(f"\nAnswer:\n{response2['answer']}")

# Extract 3GPP references
ts_refs = re.findall(r'TS\s*(\d{2}\.\d{3})', response2['answer'])
tr_refs = re.findall(r'TR\s*(\d{2}\.\d{3})', response2['answer'])
if ts_refs or tr_refs:
    print(f"\n3GPP cross-references discovered:")
    for ref in sorted(set(ts_refs)):
        print(f"  TS {ref} → query the 3GPP collection for follow-up")
    for ref in sorted(set(tr_refs)):
        print(f"  TR {ref} → query the 3GPP collection for follow-up")

## 6. Question Answering

In [ ]:
QUESTION = "What is the Quality on Demand API and what QoS profiles does it support?"

retrieved = search(QUESTION, store=store, embedder=embedder, top_k=5)

from open_telco_rag.generators.openrouter import OpenRouterGenerator
generator = OpenRouterGenerator(model="meta-llama/llama-3.3-70b-instruct")

response  = generate(QUESTION, retrieved, source_type="camara", generator=generator)

print(f"Question: {QUESTION}")
print(f"{'─'*70}")
print(f"\n{response['answer']}")
print(f"\n{'─'*70}")
print(f"Sources used: {response['sources']}")
print(f"Chunks used : {response['chunks_used']}")

### 6.1 Question bank

Try the full CAMARA evaluation question set across all major API families.

In [ ]:
QUESTIONS = [
    ("What is the Quality on Demand API and what QoS profiles does it support?", "QualityOnDemand"),
    ("How does CAMARA handle network slice booking for QoS?",                    "NetworkSliceBooking"),
    ("What device location APIs does CAMARA provide?",                           "DeviceLocation"),
    ("How does the CAMARA Number Verification API work?",                        "NumberVerification"),
    ("What are the CAMARA API design guidelines for error handling?",             "Commonalities"),
    ("What does the ConnectivityInsights API provide?",                          "ConnectivityInsights"),
    ("How does the SimSwap API detect SIM swap fraud?",                          "SimSwap"),
    ("How does CAMARA define event notifications and subscriptions?",             "Commonalities"),
    ("What APIs does CAMARA provide for edge computing?",                        "SimpleEdgeDiscovery"),
    ("What is the CAMARA consent management model?",                             "IdentityAndConsentManagement"),
]

print(f"{'#':<3} {'Expected repo':<26} {'Top retrieved':<26} {'Mean sim':>9} {'Question'}")
print("-" * 100)

for i, (question, expected_repo) in enumerate(QUESTIONS, 1):
    results  = search(question, store=store, embedder=embedder, top_k=5)
    sims     = [1 - r.distance for r in results]
    mean_sim = sum(sims) / len(sims) if sims else 0
    top_src  = results[0].metadata.get("source", "").split("/")[0] if results else "?"
    hit      = "✓" if expected_repo.lower() in top_src.lower() else "✗"
    q_short  = question[:40] + "..." if len(question) > 40 else question
    print(f"{i:<3} {expected_repo:<26} {top_src[:23] + ' ' + hit:<26} {mean_sim:>9.3f} {q_short}")

## 7. Evaluation

Run the evaluation metrics across all CAMARA evaluation questions.

**Note**: metric scoring calls the LLM API (configured below).
The full run takes approximately 3–5 minutes for 10 questions.

In [ ]:
from open_telco_rag.evaluation.evaluate import evaluate_corpus
from open_telco_rag.evaluation.questions import load_questions

questions = load_questions("camara")
print(f"Loaded {len(questions)} evaluation questions for CAMARA")
print(f"Questions with reference answers: {sum(1 for q in questions if q.reference_answer)}")

In [ ]:
from open_telco_rag.evaluation._claude import make_llm_call

# RAG generation model (answers questions)
rag_llm_call = make_llm_call("openrouter", "meta-llama/llama-3.3-70b-instruct")

# Metrics evaluation model (scores faithfulness, relevancy, etc.)
llm_call = make_llm_call("openrouter", "anthropic/claude-sonnet-4.5")

result = evaluate_corpus(
    corpus       = "camara",
    store        = store,
    embedder     = embedder,
    questions    = questions,
    model        = MODEL_KEY,
    top_k        = 5,
    delay        = 1.0,
    llm_call     = llm_call,
    rag_llm_call = rag_llm_call,
)


In [ ]:
print(f"\nPer-question results — CAMARA ({MODEL_KEY}, top-k=5)")
print(f"{'#':<3} {'Faith':>6} {'Relev':>6} {'Sim':>6} {'Recall':>7} {'Correct':>8} {'Question'}")
print("-" * 95)

for i, qr in enumerate(result.question_results, 1):
    q_short = qr.question[:45] + "..." if len(qr.question) > 45 else qr.question
    print(
        f"{i:<3} "
        f"{qr.faithfulness:>6.2f} "
        f"{qr.answer_relevancy:>6.2f} "
        f"{qr.mean_similarity:>6.3f} "
        f"{max(0, qr.context_recall):>7.2f} "
        f"{max(0, qr.answer_correctness):>8.2f} "
        f"{q_short}"
    )

print("-" * 95)
print(
    f"{'AVG':<3} "
    f"{result.mean_faithfulness:>6.2f} "
    f"{result.mean_answer_relevancy:>6.2f} "
    f"{result.mean_similarity:>6.3f} "
    f"{max(0, result.mean_context_recall):>7.2f} "
    f"{max(0, result.mean_answer_correctness):>8.2f}"
)


## 8. What the metrics tell us

| Metric | Type | Interpretation |
|--------|------|----------------|
| **Faithfulness** | Generation | Are the claims in the answer grounded in the retrieved chunks? |
| **Answer relevancy** | Generation | Does the answer address what was asked? |
| **Mean similarity** | Retrieval | How confident is the retrieval? Higher = better chunk matches |
| **Context recall** | Retrieval+GT | Were the chunks needed to answer the question retrieved? |
| **Answer correctness** | Generation+GT | Is the answer factually correct vs the reference answer? |
